Summary: Silver Transformation Step

Purpose
The Silver layer transforms raw Airbnb data from the Bronze layer into clean, standardized tables ready for analysis and modelling. It enforces schema consistency, handles missing values, and applies domain-specific logic to ensure auditability and business relevance. This step is the foundation for all downstream analytics, dashboards, and ML workflows.

🛠️ What It Does
Centralized Setup
- Uses pathlib to resolve project paths dynamically, ensuring portability across environments.
- Creates the data/silver directory if it doesn’t exist, supporting reproducible execution.

Ingestion from Bronze
- Loads raw CSVs for listings, reviews, calendar, and neighbourhoods.
- Applies low_memory=False to avoid type inference issues during read.

Listings Cleaning
- Drops irrelevant columns (URLs, metadata, free-text) to reduce noise.
- Parses date fields (host_since, first_review, last_review) for temporal analysis.
- Converts price and percentage strings to numeric types.
- Encodes categorical columns for memory efficiency and modeling.
- Drops duplicates and reports row reduction.
- Saves cleaned listings to listings_clean.parquet.

Reviews Cleaning
- Removes free-text comments (no NLP planned).
- Parses review dates and ensures ID columns are numeric.
- Encodes reviewer names as categories.
- Drops duplicates and saves to reviews_clean.parquet.

Calendar Cleaning
- Parses date column and converts availability flags to boolean.
- Cleans price fields and ensures listing IDs are numeric.
- Drops duplicates and saves to calendar_clean.parquet.

QA Checks
- Assesses coverage between listings, reviews, and calendar data.
- Summarizes missing values across all tables.
- Reports timestamp ranges for temporal scope.
- Detects price outliers using IQR logic for listings and calendar data.

Failsafes Built In
- Error-tolerant parsing: Uses errors="coerce" for dates and numerics to avoid crashes.
- Column existence checks: Ensures transformations only apply to present columns.
- Duplicate handling: Drops and reports duplicates to maintain data integrity.
- Modular logic: Each cleaning step is isolated and reproducible.
- NA summaries: Provides visibility into data completeness for downstream decisions.
- Parquet output: Ensures compact, fast-loading, schema-preserving storage for all Silver tables.


Alignment with Scalable, Reproducible Pipeline

- Modularity - Each dataset is cleaned in its own block with clear logic
- Reproducibility - Uses fixed paths, deterministic transformations, and saves to versioned Parquet file
- Auditability - Reports row counts, NA summaries, and timestamp ranges
- Portability - pathlib ensures compatibility across OS and environments
- Scalability - Easily extendable to new datasets or transformations
- Business Relevance - Cleans and prepares data for dashboards, segmentation, and modeling
- Efficiency - Parquet format enables fast reads, schema retention, and compact storage

In [11]:
# ---------------------------------------------------------
# SILVER SETUP: Imports & Path Resolution
# Purpose: Centralize imports and directory setup for Silver
# layer transformations.
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
bronze_dir = project_root / "data" / "bronze"
silver_dir = project_root / "data" / "silver"
silver_dir.mkdir(parents=True, exist_ok=True)

In [12]:
# ---------------------------------------------------------
# SILVER INGESTION: Load Bronze CSVs
# Purpose: Load raw Bronze files into DataFrames for
# cleaning and standardisation.
# ---------------------------------------------------------

df_listings = pd.read_csv(bronze_dir / "listings_full.csv", low_memory=False)
df_reviews = pd.read_csv(bronze_dir / "reviews_full.csv", low_memory=False)
df_calendar = pd.read_csv(bronze_dir / "calendar.csv", low_memory=False)
df_neighbourhoods = pd.read_csv(bronze_dir / "neighbourhoods.csv")

print("✅ Loaded Bronze into DataFrames")
print("Listings:", df_listings.shape)
print("Reviews:", df_reviews.shape)
print("Calendar:", df_calendar.shape)
print("Neighbourhoods:", df_neighbourhoods.shape)

✅ Loaded Bronze into DataFrames
Listings: (94559, 79)
Reviews: (1932265, 6)
Calendar: (34512421, 7)
Neighbourhoods: (33, 2)


In [14]:
# ---------------------------------------------------------
# SILVER STEP: Clean Listings
# Purpose: Clean and standardize full listings data.
# ---------------------------------------------------------

df_listings_clean = df_listings.copy()

# Drop irrelevant columns
drop_cols = [
    "listing_url", "picture_url", "host_url",
    "host_thumbnail_url", "host_picture_url",
    "neighbourhood_group_cleansed", "calendar_updated", "license",
    "scrape_id", "last_scraped", "source", "calendar_last_scraped",
    "description", "neighborhood_overview", "name", "host_about", "neighbourhood"
]
df_listings_clean.drop(columns=drop_cols, errors="ignore", inplace=True)

# Parse dates
for col in ["host_since", "first_review", "last_review"]:
    if col in df_listings_clean.columns:
        df_listings_clean[col] = pd.to_datetime(df_listings_clean[col], errors="coerce")

# Convert price to numeric
if "price" in df_listings_clean.columns:
    df_listings_clean["price"] = (
        df_listings_clean["price"].replace(r"[\$,]", "", regex=True).astype(float)
    )

# Convert percentage strings to numeric
for col in ["host_response_rate", "host_acceptance_rate"]:
    if col in df_listings_clean.columns:
        df_listings_clean[col] = (
            df_listings_clean[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .apply(lambda x: float(x) if x.replace('.', '', 1).isdigit() else pd.NA)
            .astype("Float64")  # nullable float dtype
        )

# Convert object columns to category (except text-heavy)
text_heavy = ["amenities", "bathrooms_text"]
for col in df_listings_clean.select_dtypes(include="object").columns:
    if col not in text_heavy:
        df_listings_clean[col] = df_listings_clean[col].astype("category")

# Drop duplicates
before = len(df_listings_clean)
df_listings_clean.drop_duplicates(inplace=True)
after = len(df_listings_clean)

print(f"✅ Listings cleaned: {after:,} rows (dropped {before - after:,} duplicates)")

# Save to Silver
df_listings_clean.to_parquet(silver_dir / "listings_clean.parquet", index=False)

# NA summary
print("\nNA counts per column:")
print(df_listings_clean.isna().sum().sort_values(ascending=False).head(20))


✅ Listings cleaned: 94,559 rows (dropped 0 duplicates)

NA counts per column:
host_neighbourhood             48827
beds                           34277
bathrooms                      34244
price                          34218
estimated_revenue_l365d        34218
host_response_rate             34143
host_response_time             34143
host_acceptance_rate           27194
review_scores_location         24287
review_scores_value            24287
review_scores_checkin          24286
review_scores_communication    24263
review_scores_accuracy         24257
review_scores_cleanliness      24251
review_scores_rating           24242
last_review                    24242
reviews_per_month              24242
first_review                   24242
host_location                  22577
bedrooms                       12835
dtype: int64


In [15]:
# ---------------------------------------------------------
# SILVER STEP: Clean Reviews
# Purpose: Clean and standardize full reviews data.
# ---------------------------------------------------------

df_reviews_clean = df_reviews.copy()
df_reviews_clean.drop(columns=["comments"], errors="ignore", inplace=True)
df_reviews_clean["date"] = pd.to_datetime(df_reviews_clean["date"], errors="coerce")

for col in ["listing_id", "id", "reviewer_id"]:
    df_reviews_clean[col] = pd.to_numeric(df_reviews_clean[col], errors="coerce").astype("Int64")

if "reviewer_name" in df_reviews_clean.columns:
    df_reviews_clean["reviewer_name"] = df_reviews_clean["reviewer_name"].astype("category")

before = len(df_reviews_clean)
df_reviews_clean.drop_duplicates(inplace=True)
after = len(df_reviews_clean)

print(f"✅ Reviews cleaned: {after:,} rows (dropped {before - after:,} duplicates)")

df_reviews_clean.to_parquet(silver_dir / "reviews_clean.parquet", index=False)

print("\nNA counts per column:")
print(df_reviews_clean.isna().sum())

✅ Reviews cleaned: 1,932,265 rows (dropped 0 duplicates)

NA counts per column:
listing_id       0
id               0
date             0
reviewer_id      0
reviewer_name    2
dtype: int64


In [16]:
# ---------------------------------------------------------
# SILVER STEP: Clean Calendar
# Purpose: Clean and standardize calendar data.
# ---------------------------------------------------------

df_calendar_clean = df_calendar.copy()
df_calendar_clean["date"] = pd.to_datetime(df_calendar_clean["date"], errors="coerce")
df_calendar_clean["available"] = df_calendar_clean["available"].map({"t": True, "f": False})

for col in ["price", "adjusted_price"]:
    if col in df_calendar_clean.columns:
        df_calendar_clean[col] = (
            df_calendar_clean[col].replace(r"[\$,]", "", regex=True).astype(float)
        )

df_calendar_clean["listing_id"] = pd.to_numeric(df_calendar_clean["listing_id"], errors="coerce").astype("Int64")

before = len(df_calendar_clean)
df_calendar_clean.drop_duplicates(inplace=True)
after = len(df_calendar_clean)

print(f"✅ Calendar cleaned: {after:,} rows (dropped {before - after:,} duplicates)")

df_calendar_clean.to_parquet(silver_dir / "calendar_clean.parquet", index=False)

print("\nNA counts per column:")
print(df_calendar_clean.isna().sum())

✅ Calendar cleaned: 34,512,421 rows (dropped 0 duplicates)

NA counts per column:
listing_id               0
date                     0
available                0
price                    0
adjusted_price    34499281
minimum_nights        2103
maximum_nights        2103
dtype: int64


In [18]:
# ---------------------------------------------------------
# SILVER QA STEP: Coverage, NA Summary, Timestamp Ranges
# Purpose: Assess linkage, completeness, and temporal scope
# of Silver-layer data. Includes:
#   - Coverage checks (reviews/calendar vs listings)
#   - NA summary across all tables
#   - Timestamp range checks
#   - Basic outlier detection for price fields
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
silver_dir = project_root / "data" / "silver"

# Load Silver tables
df_listings = pd.read_parquet(silver_dir / "listings_clean.parquet")
df_reviews = pd.read_parquet(silver_dir / "reviews_clean.parquet")
df_calendar = pd.read_parquet(silver_dir / "calendar_clean.parquet")

# -------------------------------
# 🔍 Coverage Checks
# -------------------------------
listings_ids = set(df_listings["id"])
reviews_ids = set(df_reviews["listing_id"].dropna())
calendar_ids = set(df_calendar["listing_id"].dropna())

print("🧾 Listings total:", len(listings_ids))
print("🗣️ Reviews linked:", len(reviews_ids & listings_ids))
print("📅 Calendar linked:", len(calendar_ids & listings_ids))

print("Reviews coverage:", round(len(reviews_ids & listings_ids) / len(reviews_ids), 3))
print("Calendar coverage:", round(len(calendar_ids & listings_ids) / len(calendar_ids), 3))

# -------------------------------
# 📉 NA Summary
# -------------------------------
print("\n🔍 NA Summary (Top 10 columns with missing values):")

def show_na(df, name):
    na_counts = df.isna().sum()
    na_top = na_counts[na_counts > 0].sort_values(ascending=False).head(10)
    if not na_top.empty:
        print(f"\n{name}:")
        print(na_top)
    else:
        print(f"\n{name}: No missing values.")

show_na(df_listings, "Listings")
show_na(df_reviews, "Reviews")
show_na(df_calendar, "Calendar")

# -------------------------------
# 🕒 Timestamp Range Checks
# -------------------------------
print("\n🕒 Timestamp Ranges:")

def show_date_range(df, col, name):
    if col in df.columns:
        min_date = df[col].min()
        max_date = df[col].max()
        print(f"{name} → {col}: {min_date.date()} to {max_date.date()}")

show_date_range(df_listings, "host_since", "Listings")
show_date_range(df_reviews, "date", "Reviews")
show_date_range(df_calendar, "date", "Calendar")

# -------------------------------
# 💸 Price Outlier Detection
# -------------------------------
print("\n💸 Price Outlier Detection:")

def show_price_outliers(df, col, name):
    if col in df.columns:
        prices = df[col].dropna()
        q1 = prices.quantile(0.25)
        q3 = prices.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = prices[(prices < lower) | (prices > upper)]
        print(f"{name} → {col}: {len(outliers)} outliers (outside {lower:.2f}–{upper:.2f})")

show_price_outliers(df_listings, "price", "Listings")
show_price_outliers(df_calendar, "price", "Calendar")
show_price_outliers(df_calendar, "adjusted_price", "Calendar")



print("Shape of cleaned tables")
print("Listings:", df_listings_clean.shape)
print("Reviews:", df_reviews_clean.shape)
print("Calendar:", df_calendar_clean.shape)
print("Neighbourhoods:", df_neighbourhoods.shape)

🧾 Listings total: 94559
🗣️ Reviews linked: 70316
📅 Calendar linked: 94554
Reviews coverage: 1.0
Calendar coverage: 1.0

🔍 NA Summary (Top 10 columns with missing values):

Listings:
host_neighbourhood         48827
beds                       34277
bathrooms                  34244
price                      34218
estimated_revenue_l365d    34218
host_response_rate         34143
host_response_time         34143
host_acceptance_rate       27194
review_scores_location     24287
review_scores_value        24287
dtype: int64

Reviews:
reviewer_name    2
dtype: int64

Calendar:
adjusted_price    34499281
minimum_nights        2103
maximum_nights        2103
dtype: int64

🕒 Timestamp Ranges:
Listings → host_since: 2008-08-28 to 2025-03-02
Reviews → date: 2009-12-21 to 2025-03-15
Calendar → date: 2025-03-05 to 2026-03-15

💸 Price Outlier Detection:
Listings → price: 4018 outliers (outside -118.00–386.00)
Calendar → price: 3330692 outliers (outside -132.50–399.50)
Calendar → adjusted_price: 1095